In [154]:
# import modules
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
#from scipy.stats import vonmises_fisher

# load the result object
from holopy.core.holopy_object import HoloPyObject, FullLoader
from holopy.core.utils import ensure_array, dict_without
import yaml
import importlib
# load using h5py
import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy
from holopy.inference import model

In [155]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [156]:
# path to directory
DIRECTORYPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/'

# Create hologram using known ground truth parameters

In [160]:
# ground truth parameters used to generate a hologram using scattering theory
# optical parameters
medium_index = 1.33
illum_wavelen = 0.660
illum_polarization = (0.56, 0.83)
detector = hp.detector_grid(shape=100, spacing=0.177)

# geometric parameters
#
N_1_TRUE = 1.5848484802283918 #1.59
N_2_TRUE = 1.601784237444771  #1.59
R_1_TRUE = 0.6749016953839639 #0.65
R_2_TRUE = 0.6446649533295826 #0.65
# originally for spheres on top of each other, changed to less problematic case (version 1)
# modified so at boundary between 0 and 2pi to test von Mises-Fisher (version 2)
X1_TRUE = 5.43
Y1_TRUE = 5
Z1_TRUE = 5
X2_TRUE = 4
Y2_TRUE = 5
Z2_TRUE = 5

# derived geometric parameters
Xg_TRUE = (X1_TRUE + X2_TRUE)/2
Yg_TRUE = (Y1_TRUE + Y2_TRUE)/2
Zg_TRUE = (Z1_TRUE + Z2_TRUE)/2
GAP = np.sqrt((X1_TRUE-X2_TRUE)**2+(Y1_TRUE-Y2_TRUE)**2+(Z1_TRUE-Z2_TRUE)**2)
# should use absolute value here?
THETA = np.arccos(abs(Z1_TRUE-Z2_TRUE)/GAP)
if (X1_TRUE-X2_TRUE) != 0:
    # should I be using absolute value?
    PHI = np.arctan((Y1_TRUE-Y2_TRUE)/(X1_TRUE-X2_TRUE))
elif (Y1_TRUE-Y2_TRUE) > 0:
    PHI = (np.pi)/2
elif (Y1_TRUE-Y2_TRUE) < 0:
    PHI = -np.pi/2
# both x and y are the same so phi is undefined -> set to 2pi
else:
    PHI = 2*np.pi
# adjust range from -pi to pi into 0 to 2pi
if PHI < 0:
    PHI = 2*np.pi + PHI

In [158]:
print(GAP)
print(THETA)
print(PHI)

1.4299999999999997
1.5707963267948966
0.0


In [6]:
# create a hologram from two spheres, with parameters specified above
s1 = Sphere(center=(X1_TRUE, Y1_TRUE, Z1_TRUE), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(X2_TRUE, Y2_TRUE, Z2_TRUE), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo1 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
# need to specify noise sd if priors not uniform, here use one from Caroline's fit used
# in mcmc fitting notebook
# specifying it in this way seems to lead to errors when loading fits as it appears as
# a coordinate instead of an attribute, but assigning as attrs here also leads to errors
# easiest to assign as coord here and then deal with downstream
# holo1.assign_attrs(noise_sd = 0.00862558)
holo1['noise_sd'] = 0.00862558 
hp.show(holo1)

# save path for side by side geometry
GEOMETRYPATH = DIRECTORYPATH+'phi_boundary_side_by_side_geometry/'

Some other geometries that I could use are below

In [ ]:
# create a hologram from two spheres, one above the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(5, 5, 3.5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_over = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_over['noise_sd'] = 0.00862558 
hp.show(holo_over)

In [7]:
# create a hologram from two spheres, one next to the other (version 1)
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 4, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_side['noise_sd'] = 0.00862558 
hp.show(holo_side)

In [ ]:
# create a hologram from two spheres next to each other at phi boundary (version 2)
s1 = Sphere(center=(5.43, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 5, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo_side = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
holo_side['noise_sd'] = 0.00862558 
hp.show(holo_side)

# Set parameters used in model fitting/ initialization best guesses

In [159]:
# info for modelling/ fitting

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957
# not sure what this is and if I should be changing it or not
DIMER_Z_GUESS = 4.20

# Test dummy parameter method where theta and phi are combined in KaiModel object

In [8]:
# define model creation using KaiModel object
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return model.KaiModel(scatterer, alpha=alpha)

## Skip CMA for now and so hold off on implementing model.generate_guess

In [11]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
# recently changed R_1_Mean to R_1_TRUE and same for R2, N1, N2
r_1 = prior.BoundedGaussian(R_1_TRUE, R_1_SIGMA, lower_bound=0, upper_bound=1.0, name="r_1")
r_2 = prior.BoundedGaussian(R_2_TRUE, R_2_SIGMA, lower_bound=0, upper_bound=1.0, name="r_2")
n_1 = prior.BoundedGaussian(N_1_TRUE, N_1_SIGMA, lower_bound=0, upper_bound=1.7, name="n_1")
n_2 = prior.BoundedGaussian(N_2_TRUE, N_2_SIGMA, lower_bound=0, upper_bound=1.7, name="n_2")
x_g = prior.BoundedGaussian(x, SPACING, lower_bound=(x-5), 
                            upper_bound=(x+5), name="x_g")
y_g = prior.BoundedGaussian(y, SPACING, lower_bound=(y-5), 
                            upper_bound=(y+5), name="y_g")
z_g = prior.BoundedGaussian(Zg_TRUE, 1, lower_bound=0, upper_bound=50, name="z_g")
# note: k argument needs to be the same for both theta and phi
theta = prior.Theta(5, THETA, name="theta")
phi = prior.Phi(5, PHI, name="phi")
gap = prior.BoundedGaussian((GAP-R_1_TRUE-R_2_TRUE), 0.005, lower_bound=0, 
                            upper_bound=R_1_TRUE, name="gap")
# changed from 0.8 to 0.997
alpha = prior.BoundedGaussian(0.997, 0.5, lower_bound=0.5, 
                            upper_bound=1.2, name="alpha") 
step_5_parameters = {'r_1': r_1, 'r_2': r_2, 'n_1': n_1, 'n_2': n_2,
                    'x_g': x_g, 'y_g': y_g, 'z_g': z_g,
                    'theta': theta, 'phi': phi, 'gap': gap, 'alpha': alpha}
model5 = create_kaimodel(step_5_parameters)

In [12]:
model5._parameters

[BoundedGaussian(mu=1.59, sd=0.00027320846407879903, lower_bound=0, upper_bound=1.7, name='n_1'),
 BoundedGaussian(mu=0.65, sd=0.00047233977835876834, lower_bound=0, upper_bound=1.0, name='r_1'),
 BoundedGaussian(mu=4.715, sd=0.177, lower_bound=-0.28500000000000014, upper_bound=9.715, name='x_g'),
 BoundedGaussian(mu=0.12999999999999967, sd=0.005, lower_bound=0, upper_bound=0.65, name='gap'),
 Phi(mu=0.0, name='phi', sd=1),
 Theta(mu=1.5707963267948966, name='theta', sd=1),
 BoundedGaussian(mu=5.0, sd=0.177, lower_bound=0.0, upper_bound=10.0, name='y_g'),
 BoundedGaussian(mu=5.0, sd=1, lower_bound=0, upper_bound=50, name='z_g'),
 BoundedGaussian(mu=1.59, sd=0.0003353994780766957, lower_bound=0, upper_bound=1.7, name='n_2'),
 BoundedGaussian(mu=0.65, sd=0.000617682113114549, lower_bound=0, upper_bound=1.0, name='r_2'),
 BoundedGaussian(mu=0.997, sd=0.5, lower_bound=0.5, upper_bound=1.2, name='alpha')]

## Start by just using the true values as starting points to make sure fits are working as expected

In [16]:
# now generate fit strategy with initial points given by the true values

# originally 50 walkers but start with 30 for speed
nwalkers = 30

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    for p in model5._parameters:
        means.append(p.mu)
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy_initial_match_true = EmceeStrategy(npixels=8000, nwalkers=nwalkers, walker_initial_pos=initial_guess)

In [20]:
initial_guess

array([[1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.78539816,
        1.57079633, 4.5       , 5.        , 1.59      , 0.65      ,
        0.997     ],
       [1.59      , 0.65      , 4.5       , 0.11421356, 0.

In [17]:
# save path for fit with initial conditions given by ground truth values
INITIALCONDPATH = GEOMETRYPATH + 'initial_conditions_match_true_values/'
SAVEPATH = INITIALCONDPATH + 'von_Mises_Fisher_fit_with_alpha_corrected_1'

## Try to implement reasonable starting points

In [126]:
# now try to actually fit

# originally 50 walkers but start with 30 for speed
nwalkers = 30
nsamples = 2000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    for p in model5._parameters:
        if p.name == 'theta':
            means.append(p.mu + np.random.normal(0,p.sd*0.05))
        elif p.name == 'phi':
            means.append(p.mu + np.random.normal(0,p.sd*0.05))
        elif p.name == 'x_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'y_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'z_g':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'alpha':
            means.append(p.mu + np.random.normal(0,p.sd*1))
        elif p.name == 'gap':
            means.append(p.mu + np.random.normal(0,p.sd*5))
        else:
            means.append(p.mu + np.random.normal(0,p.sd*50))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

In [22]:
#for p in model5._parameters:
    #print(p.name)
    #print(p.mu)
    #print(p.sd)

n_1
1.59
0.00027320846407879903
r_1
0.65
0.00047233977835876834
x_g
4.5
0.177
gap
0.1142135623730951
0.005
phi
0.7853981633974483
1
theta
1.5707963267948966
1
y_g
4.5
0.177
z_g
5.0
1
n_2
1.59
0.0003353994780766957
r_2
0.65
0.000617682113114549
alpha
0.997
0.5


In [40]:
initial_guess

array([[1.60137473, 0.62013099, 4.5834806 , 0.10919162, 0.76812068,
        1.57192617, 4.64605481, 5.61875646, 1.55261772, 0.65162255,
        1.32315792],
       [1.57903361, 0.65316845, 4.37746848, 0.12597718, 0.78699211,
        1.53284131, 4.41520574, 4.32332338, 1.56506646, 0.6659925 ,
        1.06127354],
       [1.61533141, 0.64345918, 4.36806311, 0.05707434, 0.79367722,
        1.52547733, 4.40384919, 5.50559799, 1.57635642, 0.62993696,
        0.95381779],
       [1.58278194, 0.62745609, 4.61504633, 0.15135283, 0.72418995,
        1.60935489, 4.33569894, 4.59425242, 1.55090014, 0.68797486,
        1.51632315],
       [1.5786116 , 0.67003436, 4.49167146, 0.12257201, 0.74661344,
        1.5530907 , 4.2461472 , 4.3752894 , 1.57628246, 0.59635332,
        1.6116085 ],
       [1.61104068, 0.68248025, 4.61056152, 0.12920485, 0.73976576,
        1.54054156, 4.69348964, 5.1350506 , 1.56865228, 0.68479721,
        1.0321092 ],
       [1.62379044, 0.64355382, 4.55990225, 0.09049225, 0.

In [127]:
print(nsamples)
print(emcee_strategy)

2000
EmceeStrategy(nwalkers=30, nsamples=2000, npixels=8000, walker_initial_pos=array([[1.5815176 , 0.61377595, 4.53266433, 0.11244709, 0.71509768,
        1.60547219, 4.80063279, 5.12228938, 1.56553857, 0.62286946,
        1.04036161],
       [1.56702118, 0.68125043, 4.37671981, 0.12174153, 0.79831997,
        1.45931005, 4.67003165, 5.04516877, 1.62941401, 0.62297778,
        1.89483459],
       [1.56994468, 0.62471641, 4.21383502, 0.13935597, 0.73718282,
        1.43948836, 4.49290464, 4.54114358, 1.60059889, 0.61070015,
        0.23742607],
       [1.58128283, 0.66685879, 4.57580109, 0.10683535, 0.82384398,
        1.54218801, 4.6361506 , 5.75531085, 1.58906766, 0.66373151,
        0.88259569],
       [1.59044884, 0.62341989, 4.52243856, 0.13501159, 0.71685787,
        1.59702019, 4.57456486, 4.34639876, 1.56521241, 0.67205089,
        0.7911461 ],
       [1.5931867 , 0.64124566, 4.7958348 , 0.13919736, 0.89375329,
        1.55736025, 4.54191755, 5.14975216, 1.60193501, 0.66240995,

In [134]:
# save path for random starting conditions
INITIALCONDPATH = GEOMETRYPATH + 'variable_initial_conditions_version_1/tuned_initial_condition_variance/'
NAME_OF_FIT = 'von_Mises_Fisher_nsample_2000_fit_1'
SAVEPATH = INITIALCONDPATH + NAME_OF_FIT

In [129]:
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/tuned_initial_condition_variance/von_Mises_Fisher_nsample_2000_fit_1


## Tuned initial starting conditions to match Caroline's code

In [13]:
# now try to actually fit

# originally 50 walkers but start with 30 for speed
nwalkers = 30
nsamples = 1000 # default is 1000

# originally used model5.generate_guess(nwalkers, scaling=0.1) but need different method
# before we implement .generate_guess for joint von Mises_Fisher
initial_guess = np.zeros((nwalkers, len(model5._parameters)))
for n in range(nwalkers):
    means = []
    # add some variance in starting point based on variance in distributions
    # need to make sure to avoid unphysical starting positions
    # added more variance to some and less to others (angles) to hopefully get faster convergence
    # originally it was p.sd*0.5
    scaling = 0.1
    for p in model5._parameters:
        if p.name == 'theta':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        elif p.name == 'phi':
            means.append(p.mu + scaling*(np.random.normal(p.mu,0.1)-p.mu))
        #elif p.name == 'x_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'y_g':
            #means.append(p.mu + scaling*np.random.normal(0,0.177))
        #elif p.name == 'z_g':
            #means.append(p.mu + scaling*np.random.normal(0,1))
        else:
            means.append(p.mu + scaling*(np.random.normal(p.mu,p.sd)-p.mu))
    # correct phi and theta for continuous angular space ie) 0 to 2pi and 0 to pi
    means[4] = means[4]%(2*np.pi)
    means[5] = means[5]%np.pi
    initial_guess[n,:] = means
emcee_strategy = EmceeStrategy(npixels=8000, nwalkers=nwalkers,nsamples=nsamples, walker_initial_pos=initial_guess)

In [14]:
initial_guess

array([[1.58998484e+00, 6.50033563e-01, 4.68471376e+00, 1.29725155e-01,
        1.89257244e-02, 1.59253098e+00, 5.03282400e+00, 5.02209085e+00,
        1.59004545e+00, 6.50097647e-01, 9.68611506e-01],
       [1.59003380e+00, 6.49888265e-01, 4.70714607e+00, 1.29970519e-01,
        4.07132429e-03, 1.57191913e+00, 4.99385583e+00, 5.11451417e+00,
        1.59001061e+00, 6.50063860e-01, 9.88968903e-01],
       [1.58997419e+00, 6.50076721e-01, 4.69642902e+00, 1.29729900e-01,
        1.24888985e-02, 1.55763037e+00, 5.00028634e+00, 4.96210204e+00,
        1.59001736e+00, 6.49957069e-01, 9.47020380e-01],
       [1.58998765e+00, 6.50015646e-01, 4.72284828e+00, 1.29832795e-01,
        6.26628103e+00, 1.57505282e+00, 4.99018115e+00, 5.03997719e+00,
        1.58997010e+00, 6.49911837e-01, 1.01110733e+00],
       [1.58999940e+00, 6.49964797e-01, 4.70603410e+00, 1.29212475e-01,
        2.77735502e-04, 1.55945504e+00, 5.01148667e+00, 4.82256228e+00,
        1.58998602e+00, 6.49958365e-01, 9.56585954e-

In [15]:
print(nsamples)
print(emcee_strategy)

1000
EmceeStrategy(nwalkers=30, nsamples=1000, npixels=8000, walker_initial_pos=array([[1.58998484e+00, 6.50033563e-01, 4.68471376e+00, 1.29725155e-01,
        1.89257244e-02, 1.59253098e+00, 5.03282400e+00, 5.02209085e+00,
        1.59004545e+00, 6.50097647e-01, 9.68611506e-01],
       [1.59003380e+00, 6.49888265e-01, 4.70714607e+00, 1.29970519e-01,
        4.07132429e-03, 1.57191913e+00, 4.99385583e+00, 5.11451417e+00,
        1.59001061e+00, 6.50063860e-01, 9.88968903e-01],
       [1.58997419e+00, 6.50076721e-01, 4.69642902e+00, 1.29729900e-01,
        1.24888985e-02, 1.55763037e+00, 5.00028634e+00, 4.96210204e+00,
        1.59001736e+00, 6.49957069e-01, 9.47020380e-01],
       [1.58998765e+00, 6.50015646e-01, 4.72284828e+00, 1.29832795e-01,
        6.26628103e+00, 1.57505282e+00, 4.99018115e+00, 5.03997719e+00,
        1.58997010e+00, 6.49911837e-01, 1.01110733e+00],
       [1.58999940e+00, 6.49964797e-01, 4.70603410e+00, 1.29212475e-01,
        2.77735502e-04, 1.55945504e+00, 5.01

In [9]:
# save path for random starting conditions
INITIALCONDPATH = GEOMETRYPATH + 'variable_initial_conditions_version_1/tuned_initial_condition_variance_to_match_Caroline/'
NAME_OF_FIT = 'von_Mises_Fisher_nsample_1000_fit_1'
SAVEPATH = INITIALCONDPATH + NAME_OF_FIT

In [17]:
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/phi_boundary_side_by_side_geometry/variable_initial_conditions_version_1/tuned_initial_condition_variance_to_match_Caroline/von_Mises_Fisher_nsample_1000_fit_1


## Run fit

In [18]:
# make sure to change strategy to desired one
results5 = hp.sample(dimer_holo, model5, strategy=emcee_strategy)
hp.save(SAVEPATH+'_mcmc.h5', results5)

print('von Mises-Fisher angles fit completed')
print(results5.guess_parameters)
print(results5.parameters)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/xarray/core/common.py:615: FutureWarning: Updating MultiIndexed coordinate 'point' would corrupt indices for other variables: ['x', 'y', 'z']. This will raise an error in the future. Use `.drop_vars({'x', 'point', 'y', 'z'})` before assigning new coordinate values.
  data.coords.update(results)


von Mises-Fisher angles fit completed
{'n_1': 1.59, 'r_1': 0.65, 'x_g': 4.715, 'gap': 0.12999999999999967, 'phi': 0.0, 'theta': 1.5707963267948966, 'y_g': 5.0, 'z_g': 5.0, 'n_2': 1.59, 'r_2': 0.65, 'alpha': 0.997}
{'n_1': 1.5899660246261276, 'r_1': 0.6499631403504971, 'x_g': 4.714990893346477, 'gap': 0.1297681430446709, 'phi': 2.9938658519761748e-05, 'theta': 1.5707082917612614, 'y_g': 4.9999833649565595, 'z_g': 4.999991514574948, 'n_2': 1.589974418762824, 'r_2': 0.6500163219743975, 'alpha': 1.0001319502557136}


In [13]:
# return real fit values
starting_means = []
for p in model5._parameters:
        starting_means.append(p.mu)
print(starting_means)

[1.59, 0.65, 4.715, 0.12999999999999967, 0.0, 1.5707963267948966, 5.0, 5.0, 1.59, 0.65, 0.997]


In [14]:
print(dimer_holo.noise_sd)

<xarray.DataArray 'noise_sd' ()>
array(0.00862558)
Coordinates:
    noise_sd  float64 0.008626


In [31]:
# need scipy 1.15 while we have 1.10 is this new version incompatible? -> yes incompatible with parrellel tempering
mu = np.array([-np.sqrt(0.5), -np.sqrt(0.5), 0])
vmf = stats.vonmises_fisher(mu, 5)

AttributeError: module 'scipy.stats' has no attribute 'vonmises_fisher'

In [39]:
-11.425662431912794%(2*np.pi)

1.140708182446378

# Test different geometries

## Test with spheres side by side

## Test with spheres very slightly offset from one above the other

# Visualize fit results

## Load fit if necessary

In [27]:
# path that determines what fit you load
results_path = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/normal_side_by_side_geometry/variable_initial_conditions_version_1/longer_fits/von_Mises_Fisher_nsample_2000_fit_1_mcmc.h5'

In [16]:
INITIALCONDPATH = GEOMETRYPATH + 'variable_initial_conditions_version_1/particle_swap_not_degenerate/'

In [17]:
# alternative way of getting results path (less explicit)
LOAD_NAME_OF_FIT = 'von_Mises_Fisher_nsample_1000_fit_1'
results_path = INITIALCONDPATH + LOAD_NAME_OF_FIT + '_mcmc.h5'

In [18]:
# try to load fit result object using hp.load code
# may need to modify code now that noise_sd is assigned as an attribute instead of a coord
attr_coords = '_attr_coords'
def unpack_attrs(a):
    if len(a) == 0:
        return a
    new_attrs={}
    attr_ref = yaml.load(a[attr_coords], Loader=FullLoader)
    attrs_to_ignore = ['spacing', 'name', '_dummy_channel', '_image_scaling']
    for attr in dict_without(attr_ref, attrs_to_ignore):
        if attr_ref[attr]:
            new_attrs[attr] = xr.DataArray(
                a[attr],
                coords=attr_ref[attr],
                dims=list(attr_ref[attr].keys()))
        elif attr in a:
            new_attrs[attr] = yaml.safe_load(a[attr])
        else:
            new_attrs[attr] = None
    return new_attrs

with xr.open_dataset(results_path, engine='h5netcdf') as ds:
    if '_source_class' in ds.attrs:
        _source_class = ds.attrs.pop('_source_class')
        pathtok = _source_class.split('.')
        cls = getattr(importlib.import_module(".".join(pathtok[:-1])), pathtok[-1])
        #ds.close()
        #return_variable = cls._load(results_path)
    #def _load(cls, ds, **kwargs):
        #with xr.open_dataset(ds, engine='h5netcdf', **kwargs) as ds:
        print(ds.load())
        dataset = ds
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        # Kai added this to move noise_sd to correct place
        data.attrs['noise_sd'] = data.coords['noise_sd'].to_numpy()
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            # seems like noise_sd should be attribute not coordinate
            coordnames.remove('noise_sd') # added this
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
            print(data)
        print(dataset.attrs['model'])
        # seems like model is a problem because I have an sd attribute in the priors?
        #model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])
        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        # return args
        args = outlist
        
        #args = cls._unserialize(ds.load())
        return_variable = cls(*args)
'''
def _unserialize(cls, dataset):
        data = dataset.data
        data.attrs = unpack_attrs(data.attrs)
        if '_flat' in data.attrs.keys():
            flats = np.array(data.attrs['_flat']).T
            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]
            codes = [[level.index(f) for f in flat]
                     for level, flat in zip(levels, flats)]
            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])
            coordnames = list(data.coords)
            coordnames.remove('point')
            coords = {coord: data[coord] for coord in coordnames}
            coords['flat'] = flat_index
            data = xr.DataArray(data.values, dims=coordnames + ['flat'],
                                coords=coords, attrs=data.attrs)
        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)
        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)
        outlist = [data, model, strategy]
        outlist.append(yaml.safe_load(dataset.attrs['time']))
        kwargs = yaml.safe_load(dataset.attrs['_kwargs'])

        for key in ['lnprobs', 'samples', '_best_fit']:
            try:
                kwargs[key] = getattr(dataset, key)
                kwargs[key].attrs = unpack_attrs(kwargs[key].attrs)
            except AttributeError:
                pass
        outlist.append(kwargs)
        return outlist
'''

<xarray.Dataset>
Dimensions:    (point: 8000, walker: 30, chain: 1000, parameter: 11)
Coordinates:
    noise_sd   float64 0.008626
  * point      (point) int64 0 1 2 3 4 5 6 ... 7994 7995 7996 7997 7998 7999
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Dimensions without coordinates: walker, chain
Data variables:
    data       (point) float64 1.023 1.026 0.9914 0.9978 ... 0.9833 0.9064 1.002
    lnprobs    (walker, chain) float64 -7.674e+03 -7.674e+03 ... 3.07e+04
    samples    (walker, chain, parameter) float64 1.59 0.65 ... 0.6501 0.9998
Attributes:
    model:     !KaiModel\n_dummy_scatterer: !Spheres\n  scatterers: [!Sphere ...
    strategy:  !EmceeStrategy\nnwalkers: 30\nnsamples: 1000\nnpixels: 8000\nw...
    time:      1139.8134019374847
    _kwargs:   {}\n
<xarray.DataArray (flat: 8000)>
array([1.02280054, 1.02571659, 0.99141749, ..., 0.98331462, 0.90639182,
       1.00177476])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat

"\ndef _unserialize(cls, dataset):\n        data = dataset.data\n        data.attrs = unpack_attrs(data.attrs)\n        if '_flat' in data.attrs.keys():\n            flats = np.array(data.attrs['_flat']).T\n            levels = [data.original_dims[key] for key in ['x', 'y', 'z']]\n            codes = [[level.index(f) for f in flat]\n                     for level, flat in zip(levels, flats)]\n            flat_index = pd.MultiIndex(levels, codes, names=['x', 'y', 'z'])\n            coordnames = list(data.coords)\n            coordnames.remove('point')\n            coords = {coord: data[coord] for coord in coordnames}\n            coords['flat'] = flat_index\n            data = xr.DataArray(data.values, dims=coordnames + ['flat'],\n                                coords=coords, attrs=data.attrs)\n        model = yaml.load(dataset.attrs['model'], Loader=FullLoader)\n        strategy = yaml.load(dataset.attrs['strategy'], Loader=FullLoader)\n        outlist = [data, model, strategy]\n     

In [19]:
# seems like don't need model at least for current analysis
# end up with noise_sd as a coordinate which is weird
print(return_variable)
samples = return_variable.samples[:,999]
lnprob = return_variable.lnprobs[5]
# can get rid of noise_sd coord using .reset_coords('noise_sd', drop = True)
print(samples.reset_coords('noise_sd',drop=True))
print(lnprob.reset_coords('noise_sd',drop=True))
burnt_samples = return_variable.burn_in(110).samples[:,889]

SamplingResult(data=<xarray.DataArray (flat: 8000)>
array([1.02280054, 1.02571659, 0.99141749, ..., 0.98331462, 0.90639182,
       1.00177476])
Coordinates:
  * flat     (flat) object MultiIndex
  * x        (flat) float64 9.381 7.08 1.416 6.018 ... 15.4 12.92 9.027 12.04
  * y        (flat) float64 13.1 12.92 15.04 9.204 ... 4.248 3.009 6.372 14.34
  * z        (flat) int64 0 0 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0 0
Attributes:
    _flat:               [[9.381, 13.097999999999999, 0], [7.08, 12.921, 0], ...
    illum_polarization:  <xarray.DataArray (vector: 3)>\narray([0.55930131, 0...
    illum_wavelen:       0.66
    medium_index:        1.33
    noise_sd:            0.00862558
    original_dims:       {'x': [0.0, 0.177, 0.354, 0.5309999999999999, 0.708,..., model=<module 'holopy.inference.model' from '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/model.py'>, strategy=EmceeStrategy(nwalkers=30, nsamples=1000, npixels=8000, w

In [20]:
# set results equal to loaded fit for further analysis
results5 = return_variable

## Work on futher data processing (ie drop pre-burn in data, drop non-converged fits, and decimate remaining data so its independent)

In [21]:
# set save path for figures (only necessary if loaded fit and not from earlier)
SAVEPATH = INITIALCONDPATH + LOAD_NAME_OF_FIT
print(SAVEPATH)

/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/phi_boundary_side_by_side_geometry/variable_initial_conditions_version_1/tuned_initial_condition_variance_to_match_Caroline/von_Mises_Fisher_nsample_1000_fit_1


In [85]:
samples = results5.samples
print(samples[:,999][0])
print(means)

<xarray.DataArray 'samples' (parameter: 11)>
array([1.59006508, 0.64988053, 4.71501682, 0.1299582 , 6.28348737,
       1.57118209, 4.99974836, 5.00007137, 1.59001466, 0.65006281,
       0.99978849])
Coordinates:
    noise_sd   float64 0.008626
  * parameter  (parameter) object 'n_1' 'r_1' 'x_g' ... 'n_2' 'r_2' 'alpha'
Attributes:
    acceptance_fraction:  0.17436666666666667


NameError: name 'means' is not defined

In [91]:
print(samples[:,0].sel(parameter='phi'))

<xarray.DataArray 'samples' (walker: 30)>
array([6.30146403e+00, 6.28725663e+00, 6.29567421e+00, 6.26628103e+00,
       6.28346304e+00, 6.29883313e+00, 6.29254765e+00, 6.30729340e+00,
       6.29386983e+00, 6.27615092e+00, 6.28262424e+00, 6.29437686e+00,
       6.29943119e+00, 6.29375981e+00, 6.28865009e+00, 6.27536425e+00,
       6.28068113e+00, 6.27477939e+00, 6.27191841e+00, 3.66191249e-03,
       2.84605565e-03, 1.67081942e-02, 6.26623323e+00, 8.56338259e-03,
       1.15911830e-02, 6.26921695e+00, 1.99474681e-02, 1.29390939e-02,
       9.46315183e-03, 8.77535979e-03])
Coordinates:
    noise_sd   float64 0.008626
    parameter  <U3 'phi'
Dimensions without coordinates: walker
Attributes:
    acceptance_fraction:  0.17436666666666667


In [197]:
print(initial_guess[0])

[1.59001602 0.64999016 4.50681305 0.11351432 0.78324487 1.5647065
 4.46111155 4.97778223 1.59001142 0.65002332 1.02401675]


In [23]:
# select gap parameter values for first walker (all chains)
gaps_example = samples[1].sel(parameter='gap')
print(gaps_example)

<xarray.DataArray 'samples' (chain: 1000)>
array([0.12997052, 0.12997052, 0.12997052, 0.12997052, 0.12997052,
       0.12997052, 0.12997052, 0.12997052, 0.12997052, 0.12997052,
       0.12997052, 0.12997052, 0.12997052, 0.12997052, 0.12997052,
       0.12997589, 0.12998106, 0.12998106, 0.12998106, 0.13001432,
       0.1300349 , 0.1300349 , 0.1300349 , 0.13001131, 0.12999091,
       0.12999091, 0.12999091, 0.12999526, 0.12999526, 0.12999526,
       0.12999526, 0.12999526, 0.12999526, 0.1299985 , 0.1299985 ,
       0.1299985 , 0.1299985 , 0.1299985 , 0.12996656, 0.12996656,
       0.12996656, 0.12996656, 0.12996656, 0.12996656, 0.12996656,
       0.12996656, 0.12996338, 0.12996338, 0.12996338, 0.12996338,
       0.12995477, 0.12995477, 0.12995477, 0.12995477, 0.12995477,
       0.1299664 , 0.1299664 , 0.1299664 , 0.1299664 , 0.1299664 ,
       0.1299664 , 0.1299664 , 0.12993532, 0.12993532, 0.12993532,
       0.12993532, 0.12993532, 0.12991296, 0.12991296, 0.12991296,
       0.12987339, 

In [24]:
plt.figure()
plt.plot(gaps_example)

### Drop pre-burn in (right now ad hoc but come back to make more systematic)

In [25]:
# plot pre-burn in
plt.figure()
plt.title('Pre burn-in logprob of fit')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(11):
    plt.plot(results5.lnprobs[i])
plt.savefig(SAVEPATH + '/pre_burn_in_lnprob.png')

In [164]:
for i in range(4):
    #plt.plot(results5.lnprobs[i])
    print(results5.lnprobs[:,999][i])

<xarray.DataArray ()>
array(30472.66661571)
Attributes:
    acceptance_fraction:  0.3131166666666666
<xarray.DataArray ()>
array(-inf)
Attributes:
    acceptance_fraction:  0.3131166666666666
<xarray.DataArray ()>
array(30473.05729672)
Attributes:
    acceptance_fraction:  0.3131166666666666
<xarray.DataArray ()>
array(30465.00812573)
Attributes:
    acceptance_fraction:  0.3131166666666666


In [104]:
# Use .burn_in() to chop off data before a specific sample number
cut_number = 300
burnt_results5 = results5.burn_in(cut_number) 
#120 seems good for 1000 somewhat random start
#300 for 2000 chain random start
#400 seems good for 3000 chain random start
plt.figure()
plt.title(f'Post burn-in logprob (cut off first {cut_number})')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
#ids = [1,3,8,9,10] (for 3000 chain)
for i in range(30):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs_300_cutoff.png')
# to do more systematically could maybe calculate some sort of slope vs maximum slope cutoff?

In [66]:
# looking at the fits for all the different walkers it's clear that several don't converge well
plt.figure()
plt.title('Post burn-in logprob showing bad fits')
plt.ylabel('lnprob')
plt.xlabel('sample (chain)')
for i in range(30):
    plt.plot(burnt_results5.lnprobs[i])
plt.savefig(SAVEPATH + '/many_lnprobs_to_show_bad_fits.png')

In [147]:
new_sample_length = len(burnt_results5.lnprobs[i])
print(burnt_results5.lnprobs[i][new_sample_length-1])

<xarray.DataArray ()>
array(30693.09676752)
Attributes:
    acceptance_fraction:  0.3131166666666666


### Visualize Data Traces

In [30]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/many_theta_fits.png')

In [32]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/many_phi_fits.png')

In [75]:
# 18 starts at 2*pi for phi and 3 has phi off by pi
plt.figure()
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='gap'))

In [76]:
plt.figure()
plt.axhline(y=PHI, color='gray', linestyle='--', label='real value')
plt.plot(samples[3].sel(parameter='gap'))

In [74]:
plt.figure()
plt.plot(burnt_results5.lnprobs[3])

In [73]:
# plot phi mod 2*pi
plt.figure()
plt.title('Phi fit mod 2pi')
plt.ylabel('Phi mod 2pi')
plt.xlabel('sample (chain)')
two_pi = 2*np.pi
plt.axhline(y=PHI+two_pi, color='gray', linestyle='--', label='real value')
good_index = [0,1,2,4,5,6,7,8,9,11,12,13,14,18]
for i in good_index:
    phi = samples[i].sel(parameter='phi')
    for j in range(len(phi)):
        if phi[j] < 0.1:
            phi[j] = phi[j] + two_pi
    plt.plot(phi)
plt.savefig(SAVEPATH + '/few_phi_mod_2*pi_fits.png')

In [34]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/few_gap_fits.png')

In [36]:
# look at some traces of r1
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/many_r1_fits.png')

In [37]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/few_r2_fits.png')

In [39]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in range(30):
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/many_n1_fits.png')

In [40]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in range(4):
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/few_n2_fits.png')

In [40]:
print(samples[i].sel(parameter='n_2'))

<xarray.DataArray 'samples' (chain: 2000)>
array([1.58986979, 1.58986979, 1.58986979, ..., 1.59005263, 1.59005263,
       1.59005263])
Coordinates:
    noise_sd   float64 0.008626
    parameter  <U3 'n_2'
Dimensions without coordinates: chain
Attributes:
    acceptance_fraction:  0.30376666666666663


In [51]:
# look at some of the traces for walkers that don't converge
plt.plot(samples[7].sel(parameter='theta'))
plt.plot(samples[10].sel(parameter='theta'))

In [46]:
# look at theta that did converge and add pi to them
for i in range(4):
    plt.plot(samples[i].sel(parameter='theta')+np.pi)

In [54]:
plt.plot(samples[7].sel(parameter='phi'))
plt.plot(samples[10].sel(parameter='phi'))

In [57]:
plt.plot(samples[7].sel(parameter='gap'))
plt.plot(samples[10].sel(parameter='gap'))

### Drop bad convergence (ie low lnprob) walkers. Later attempt to fix these fits instead.

In [105]:
# look at distribution of final lnprob values across walkers
new_sample_length = len(burnt_results5.lnprobs[0])
plt.figure()
plt.plot(burnt_results5.lnprobs[:,(new_sample_length-1)])
plt.savefig(SAVEPATH + '/final_lnprob_values_across_walkers')
maxlnprob = max(burnt_results5.lnprobs[:,(new_sample_length-1)])
converged_value = maxlnprob - 0.02*maxlnprob
print(converged_value)

<xarray.DataArray 'lnprobs' ()>
array(30088.52279693)
Coordinates:
    noise_sd  float64 0.008626


In [99]:
# The bad fits are the swapped angles fits (visible in the reorganized pandas dataset)
# -> is there a good way to correct these or should I just drop them?
# Start by implementing a cutoff in lnprobs to drop them and then can work on fixing later
samples = burnt_results5.samples
converged_samples_nan = xr.DataArray()
# 0 doesn't work as a cutoff universally, example, 3000 chain fit 1 needs 15000 as cutoff
bad_fit_index = []
good_fit_index = []
for i in range(len(samples)):
    new_sample_length = len(burnt_results5.lnprobs[i])
    if burnt_results5.lnprobs[i][new_sample_length-1] > converged_value:
        converged_samples_nan = xr.concat([converged_samples_nan,samples[i]],'walker')
        # .append() isn't quite what we want, try to use xarray methods
        good_fit_index.append(i)
    else:
        bad_fit_index.append(i)
    converged_samples = converged_samples_nan[1:]
print(converged_samples)
print(len(converged_samples))
print(bad_fit_index)

<xarray.DataArray (walker: 28, chain: 880, parameter: 11)>
array([[[1.58997018, 0.65000668, 4.71486485, ..., 1.58994219,
         0.6498757 , 0.99730862],
        [1.58996951, 0.65000693, 4.71508088, ..., 1.58994302,
         0.64987764, 0.99731007],
        [1.58996949, 0.65000687, 4.71508543, ..., 1.58994288,
         0.6498774 , 0.99736523],
        ...,
        [1.59009134, 0.64983266, 4.71506223, ..., 1.59003621,
         0.65014018, 0.99968251],
        [1.59006508, 0.64988053, 4.71501682, ..., 1.59001466,
         0.65006281, 0.99978849],
        [1.59006508, 0.64988053, 4.71501682, ..., 1.59001466,
         0.65006281, 0.99978849]],

       [[1.58996753, 0.65000678, 4.71523665, ..., 1.58994611,
         0.64988255, 0.99820844],
        [1.58996753, 0.65000678, 4.71523665, ..., 1.58994611,
         0.64988255, 0.99820844],
        [1.58996753, 0.65000678, 4.71523665, ..., 1.58994611,
         0.64988255, 0.99820844],
...
        [1.58993821, 0.65011179, 4.71479744, ..., 1.589910

In [ ]:
# could also do this with ds.drop_sel(space=["IN", "IL"]) where we use walker = [drop indexes]

In [49]:
# plot converged samples
# need to modify this for each fit you want to use it for
# converged_id = [0,1,2,3,4,5,6,8,9]
plt.figure()
for id in good_fit_index:
    plt.plot(burnt_results5.lnprobs[id])

NameError: name 'converged_id' is not defined

#### Visualize good vs. bad fit parameter traces

In [106]:
# plot bad fits to see parrellels between them
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/bad_theta_fits.png')

In [107]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/bad_phi_fits.png')

In [109]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in bad_fit_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/bad_gap_fits.png')

In [110]:
# look at how many starting phi look like the starting phi that lead to these bad fits
low_start_index = []
for i in range(30):
    if (samples[i,0].sel(parameter='phi') > np.pi) and (samples[i,0].sel(parameter='phi') < 2*np.pi):
        low_start_index.append(i)
print(low_start_index)

[3, 7, 9, 10, 15, 16, 17, 18, 22, 25]


In [144]:
plt.figure()
for id in low_start_index:
    plt.plot(burnt_results5.lnprobs[id])
plt.savefig(SAVEPATH + '/low_starting_phi_lnprobs.png')

In [145]:
# look at some traces of theta
plt.figure()
plt.title('Theta fit')
plt.ylabel('Theta')
plt.xlabel('sample (chain)')
plt.axhline(y=THETA, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='theta'))
plt.savefig(SAVEPATH + '/low_starting_phi_theta_fit.png')

In [146]:
# look at some traces of phi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit.png')

In [147]:
# look at some traces of phi that are never off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi') > 4):
        plt.plot(samples[i].sel(parameter='phi'))
plt.savefig(SAVEPATH + '/low_starting_phi_phi_fit_not_off_by_pi.png')

In [121]:
# look at the end of some traces of phi that are not off by pi
plt.figure()
plt.title('Phi fit')
plt.ylabel('Phi')
plt.xlabel('sample (chain)')
plt.axhline(y=PHI+2*np.pi, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    if all(samples[i,:].sel(parameter='phi')[-50:-1] > 4):
        plt.plot(samples[i].sel(parameter='phi')[-50:-1])

In [148]:
# plot starting phi positions for low initial conditions that aren't off by mod pi
plt.figure()
plot_varible = []
for id in low_start_index:
    if samples[id,0].sel(parameter='phi') > 4:
        plot_varible.append(samples[id,0].sel(parameter='phi'))
plt.plot(plot_varible)
plt.savefig(SAVEPATH + '/low_starting_phi_plot_of_starting_phi_not_off_by_pi.png')

In [149]:
# look at some traces of gap
plt.figure()
plt.title("Gap fit")
plt.ylabel('Gap (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=GAP-R_1_TRUE-R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='gap'))
plt.savefig(SAVEPATH + '/low_starting_phi_gap_fit.png')

In [150]:
plt.figure()
plt.title("First particle radius (r1) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_r1_fit.png')

In [151]:
# look at some traces of r2
plt.figure()
plt.title("Second particle radius (r2) fit")
plt.ylabel('radius (um)')
plt.xlabel('sample (chain)')
plt.axhline(y=R_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='r_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_r2_fit.png')

In [152]:
# look at some traces of n1
plt.figure()
plt.title("First particle index of refraction (n1) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_1_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_1'))
plt.savefig(SAVEPATH + '/low_starting_phi_n1_fit.png')

In [153]:
# look at some traces of n2
plt.figure()
plt.title("Second particle index of refraction (n2) fit")
plt.ylabel('index of refraction')
plt.xlabel('sample (chain)')
plt.axhline(y=N_2_TRUE, color='gray', linestyle='--', label='real value')
for i in low_start_index:
    plt.plot(samples[i].sel(parameter='n_2'))
plt.savefig(SAVEPATH + '/low_starting_phi_n2_fit.png')

In [ ]:
# look at the fits that start with lnprob like the bad fits but then jump to good fits
# these are subset of low_start index and so already convered

### Decimate data so just keep independent fits

In [128]:
# look at autocorrelation to see how long it is before lose memory so can decimate into independent samples
series = pd.Series(converged_samples[1].sel(parameter='gap'))
autocorr_as_function_of_time = []
for i in range(len(series)):
    autocorr = series.autocorr(lag=i)
    autocorr_as_function_of_time.append(autocorr)
plt.figure()
plt.plot(autocorr_as_function_of_time)

/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/kaitorrens/miniforge3/envs/holopy-devel/lib/python3.9/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [167]:
# look at what corresponding trace looks like
plt.figure()
plt.plot(converged_samples[1].sel(parameter='gap'))

In [129]:
# look at autocorrelation of gap data from different walkers
plt.figure()
plt.title('autocorrelation of gap')
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='gap'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)
plt.savefig(SAVEPATH + '/auto_correlation_of_gap.png')

In [221]:
# look at trace of gap from different walkers
plt.figure()
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='gap'))

In [130]:
# look at autocorrelation of theta data from different walkers
plt.figure()
for n in range(len(converged_samples)):
    series = pd.Series(converged_samples[n].sel(parameter='theta'))
    autocorr_as_function_of_time = []
    for i in range(len(series)):
        autocorr = series.autocorr(lag=i)
        autocorr_as_function_of_time.append(autocorr)
    plt.plot(autocorr_as_function_of_time)

In [171]:
# look at trace of theta from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='theta'))

In [185]:
# look at trace of n_1 from different walkers
plt.figure
for n in range(len(converged_samples)):
    plt.plot(converged_samples[n].sel(parameter='n_1'))

In [103]:
# We can use the following notation to cycle throught the different parameter labels
for parameter in converged_samples.coords['parameter'].data:
    print(parameter)

n_1
r_1
x_g
gap
phi
theta
y_g
z_g
n_2
r_2
alpha


In [131]:
# Plot autocorrelation for all the different parameters
# set up plotting
number_columns = int(np.ceil(len(converged_samples.coords['parameter'].data)/3))
fig,axes = plt.subplots(3,number_columns)
m = 1
# go through analysis
for parameter_name in converged_samples.coords['parameter'].data:
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
        plt.subplot(3,number_columns,m)
        plt.plot(autocorr_as_function_of_time)
    # label plot
    row = int(np.floor((m-1)/4))
    column = int(m-4*row)
    axes[row,column-1].set_title(parameter_name)
    fig.supxlabel('Chain Number')
    fig.supylabel('Autocorrelation')
    # increment number tracker
    m = m + 1
plt.savefig(SAVEPATH + '/all_autocorrelation.png')

In [132]:
# find the correlation time for each of these parameters (ie when autocorrelation drops to 0 for the first time)
all_parameter_indices = []
for parameter_name in converged_samples.coords['parameter'].data:
    all_walker_indices = []
    for n in range(len(converged_samples)):
        series = pd.Series(converged_samples[n].sel(parameter=parameter_name))
        autocorr_as_function_of_time = []
        for i in range(len(series)):
            autocorr = series.autocorr(lag=i)
            autocorr_as_function_of_time.append(autocorr)
            if autocorr < 0:
                index = i
                all_walker_indices.append(index)
                break
            elif i == (len(series)-1):
                index = i
                all_walker_indices.append(index)
                print("sample " + str(n) + " of " +  parameter_name + " remains correlated")
    all_parameter_indices.append(all_walker_indices)
print(all_parameter_indices)
# Q: should I find max correlation time or mean correlation time for each parameter?
# start with easiest which is just overall max
overall_correlation_time = np.max(all_parameter_indices)
print(overall_correlation_time)

[[103, 419, 65, 699, 116, 71, 200, 105, 142, 587, 499, 409, 472, 198, 809, 225, 736, 341, 609, 681, 397, 332, 107, 538, 201, 465, 119, 121], [364, 183, 173, 158, 113, 144, 81, 98, 351, 61, 329, 100, 491, 257, 326, 119, 160, 66, 264, 726, 253, 785, 456, 295, 48, 202, 148, 103], [170, 138, 75, 116, 95, 520, 88, 212, 134, 154, 171, 150, 102, 147, 818, 304, 746, 194, 200, 536, 369, 103, 191, 130, 154, 208, 395, 130], [185, 263, 160, 370, 332, 330, 554, 266, 127, 719, 65, 199, 147, 73, 543, 308, 459, 131, 232, 211, 287, 747, 98, 677, 190, 170, 87, 160], [92, 183, 187, 347, 66, 138, 256, 113, 110, 12, 260, 50, 90, 90, 421, 312, 15, 178, 69, 261, 439, 98, 69, 478, 115, 211, 677, 101], [301, 97, 228, 496, 158, 128, 88, 130, 355, 162, 141, 113, 243, 174, 602, 172, 521, 72, 114, 608, 345, 798, 559, 541, 126, 424, 363, 94], [121, 301, 173, 702, 228, 210, 184, 143, 147, 160, 264, 63, 113, 145, 488, 833, 711, 220, 87, 639, 250, 587, 104, 786, 101, 124, 101, 189], [124, 120, 74, 331, 337, 378, 144, 

In [133]:
# other approach where we find the mean and then take the max
mean_parameter_corr_time = np.mean(all_parameter_indices, axis=1)
max_of_mean_parameter_corr_time = int(np.max(mean_parameter_corr_time))
print(max_of_mean_parameter_corr_time)

348


In [134]:
# now use correlation time to decimate the data
chain_number = len(converged_samples[0])
number_ind_samples_per_walker = int(np.ceil(chain_number/overall_correlation_time))
independent_samples =[]
for i in range(number_ind_samples_per_walker):
    index = (chain_number-1-overall_correlation_time*i)
    if i==0:
        independent_samples = converged_samples[:,index]
    else:
        independent_samples = xr.concat([independent_samples,(converged_samples[:,index])], 'walker')

In [135]:
# now convert independent samples into a form that can be input into seaborn pairplot
independent_samples_pd = independent_samples.to_dataframe(name = 'independent samples')
independent_samples_pd = independent_samples_pd.reset_index()
independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')
#print(independent_samples_pd)

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_73233/3190556392.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  independent_samples_pd = independent_samples_pd.pivot('walker','parameter','independent samples')


In [136]:
full_pair_plot = sns.pairplot(independent_samples_pd)
full_pair_plot.savefig(SAVEPATH + '/pair_plot_of_all_samples')

In [137]:
# seems like the parameter sets from the end of the run and the beginning of the run
# cluster in different ways, lets try seperating them

# get parameters at the end of the fitting process
end_of_run_ind = independent_samples[:len(converged_samples)]
end_of_run_ind_pd = end_of_run_ind.to_dataframe(name = 'independent samples')
end_of_run_ind_pd = end_of_run_ind_pd.reset_index()
end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
print(end_of_run_ind_pd)

# get parameters from earlier on in the fitting process (~correlation time before the end)
early_run_ind = independent_samples[len(converged_samples):]
early_run_ind_pd = early_run_ind.to_dataframe(name = 'independent samples')
early_run_ind_pd = early_run_ind_pd.reset_index()
early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')
#print(early_run_ind_pd)

parameter     alpha       gap       n_1       n_2       phi       r_1  \
walker                                                                  
0          0.999788  0.129958  1.590065  1.590015  6.283487  0.649881   
1          0.999646  0.130319  1.590093  1.589908  6.283411  0.650038   
2          1.000340  0.130163  1.590138  1.590000  6.283452  0.649881   
3          0.998614  0.129994  1.589988  1.589986  3.140951  0.650012   
4          1.000699  0.130334  1.590100  1.590025  6.283478  0.649886   
5          0.999250  0.130120  1.589899  1.589945  6.283667  0.650041   
6          0.999071  0.130091  1.589961  1.590047  6.283367  0.649877   
7          0.999604  0.130146  1.589975  1.589952  6.283417  0.650013   
8          1.001113  0.129758  1.589993  1.590038  6.283415  0.649867   
9          1.000112  0.129640  1.589743  1.589957  6.283784  0.650058   
10         0.999448  0.130321  1.590066  1.590019  6.283588  0.649912   
11         1.000409  0.130714  1.590188  1.590020  

/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_73233/2063201981.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  end_of_run_ind_pd = end_of_run_ind_pd.pivot('walker','parameter','independent samples')
/var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T/ipykernel_73233/2063201981.py:15: FutureWarning: In a future version of pandas all arguments of DataFrame.pivot will be keyword-only.
  early_run_ind_pd = early_run_ind_pd.pivot('walker','parameter','independent samples')


In [138]:
end_of_run_pair_plot = sns.pairplot(end_of_run_ind_pd)
end_of_run_pair_plot.savefig(SAVEPATH + '/pair_plot_of_end_of_run_samples')
# hmm seems like one of the fits is comparatively bad and is an outlier (for 1000 chain, random starting)

In [142]:
early_run_pair_plot = sns.pairplot(early_run_ind_pd)
early_run_pair_plot.savefig(SAVEPATH + '/30088cutoff_pair_plot_of_early_run_samples')

In [140]:
# compare average of end points of converged fits with ground truth values
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
print(mean_prediction)
print(starting_means)
mean_differences = np.zeros(len(starting_means))
corresponding_id = [2,5,8,1,4,7,9,10,3,6,0]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[i]-mean_prediction[corresponding_id[i]]
print(mean_differences)

parameter
alpha    0.999574
gap      0.130053
n_1      1.589988
n_2      1.589983
phi      3.814826
r_1      0.649975
r_2      0.650016
theta    1.571230
x_g      4.715277
y_g      5.000288
z_g      5.000169
dtype: float64
[1.59, 0.65, 4.715, 0.12999999999999967, 0.0, 1.5707963267948966, 5.0, 5.0, 1.59, 0.65, 0.997]
[ 1.15645819e-05  2.45330472e-05 -2.76560863e-04 -5.27970799e-05
 -3.81482587e+00 -4.33625043e-04 -2.88267996e-04 -1.69423960e-04
  1.65596319e-05 -1.58475337e-05 -2.57394209e-03]


In [141]:
mean_prediction = np.mean(end_of_run_ind_pd,axis=0)
mean_differences = mean_prediction.copy()
opp_corresponding_id = [10,3,0,8,4,1,9,5,2,6,7]
for i in range(len(starting_means)):
    mean_differences[i] = starting_means[opp_corresponding_id[i]]-mean_prediction[i]
print(mean_prediction)
print(mean_differences)

parameter
alpha    0.999574
gap      0.130053
n_1      1.589988
n_2      1.589983
phi      3.814826
r_1      0.649975
r_2      0.650016
theta    1.571230
x_g      4.715277
y_g      5.000288
z_g      5.000169
dtype: float64
parameter
alpha   -0.002574
gap     -0.000053
n_1      0.000012
n_2      0.000017
phi     -3.814826
r_1      0.000025
r_2     -0.000016
theta   -0.000434
x_g     -0.000277
y_g     -0.000288
z_g     -0.000169
dtype: float64


In [86]:
type(mean_differences)

pandas.core.series.Series

In [143]:
mean_differences.to_csv(SAVEPATH+'/30088cutoff_real_values_minus_mean_of_fits')